In [ ]:
import pandas as pd

# Read csv
error_data = pd.read_csv("error_data.csv")
filterflow_data = pd.read_csv("filterflow_data.csv")

# head
print("Error Data:")
print(error_data.head())

print("Filter Flow Data:")
print(filterflow_data.head())


1. Identify whether each machine has any errors.
2. Understand the time distribution of error occurrences.
3. Initially assess whether errors are related to changes in filter flow.
4. Prepare for further exploratory data analysis (EDA) and correlation modeling

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# make sure format is datetime 
filterflow_data['timestamp'] = pd.to_datetime(filterflow_data['timestamp'])
error_data['timestamp'] = pd.to_datetime(error_data['timestamp'])

# show independent graph each printer 
for printer_num in range(1, 31):
    # filter printer filterflow and order
    printer_data = filterflow_data[filterflow_data['printer'] == printer_num].sort_values('timestamp')
    
    # filter printer error data
    printer_errors = error_data[error_data['printer'] == printer_num]['timestamp']
    
    # Draw if has data
    if not printer_data.empty:
        plt.figure(figsize=(15, 6))
        
        # draw filter flow line
        plt.plot(printer_data['timestamp'], printer_data['filter_flow'], 
                label=f'Print{printer_num} Filter Flow')
        
        # label timing of error 
        for error_time in printer_errors:
            plt.plot(error_time, 0, 'rx', markersize=10)
        
        # set graph title
        plt.title(f'Print{printer_num} Filter Flow Over Time with Error Occurrences')
        plt.xlabel('Timestamp')
        plt.ylabel('Filter Flow')
        plt.legend()
        plt.grid(True)
        
        # Set x axis datatime
        ax = plt.gca()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # only show year/month/day
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        
        # rotate x label
        plt.xticks(rotation=45)
        plt.tight_layout()
    
        plt.show()
    else:
        print(f'Print{printer_num} no data')

1. Analyze the filter flow changes before and after each printer error.
2. Determine whether flow changes are correlated with the order of error occurrences, to support future prediction or maintenance.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy import stats

def analyze_correlation(printer_num):
    try:
        # select printer data
        printer_data = filterflow_data[filterflow_data['printer'] == printer_num].copy()
        printer_data['timestamp'] = pd.to_datetime(printer_data['timestamp'])
        printer_data = printer_data.sort_values('timestamp').reset_index(drop=True)
        
        printer_errors = error_data[error_data['printer'] == printer_num].copy()
        printer_errors['timestamp'] = pd.to_datetime(printer_errors['timestamp'])
        
        if printer_data.empty or printer_errors.empty:
            print(f"Print{printer_num} haven't enough data")
            return None
        
        # calculate filter_flow diff
        printer_data['flow_change'] = printer_data['filter_flow'].diff().fillna(0)
        
        # record everytime filter_flow diff when error 
        error_deltas = []
        
        window = pd.Timedelta(minutes=10)  # set range
        for error_time in printer_errors['timestamp']:
            # find data before / after
            before_error = printer_data[printer_data['timestamp'] < error_time].tail(1)
            after_error = printer_data[printer_data['timestamp'] >= error_time].head(1)
            
            if not before_error.empty and not after_error.empty:
                flow_change = after_error['filter_flow'].values[0] - before_error['filter_flow'].values[0]
                error_deltas.append({'printer': printer_num, 'error_time': error_time, 'flow_change': flow_change})
        
        # DataFrame
        if not error_deltas:
            print(f"Print{printer_num} no error")
            return None
        
        error_df = pd.DataFrame(error_deltas)
        
        # calculate Pearson & Spearman corr
        correlation_coef, p_value = stats.pearsonr(error_df['flow_change'], range(len(error_df)))
        spearman_coef, spearman_p = stats.spearmanr(error_df['flow_change'], range(len(error_df)))
        
        # print result
        print(f"\nPrint{printer_num} the flow change when error：")
        print(error_df)
        print(f"Pearson correlation: {correlation_coef:.4f}, P-value: {p_value:.4f}")
        print(f"Spearman correlation: {spearman_coef:.4f}, P-value: {spearman_p:.4f}")
        
        return error_df.assign(pearson_corr=correlation_coef, spearman_corr=spearman_coef)
    
    except Exception as e:
        print(f" Print{printer_num} no error: {str(e)}")
        return None

# Analysis
all_error_data = []
for printer_num in range(1, 31):
    result = analyze_correlation(printer_num)
    if result is not None:
        all_error_data.append(result)

if all_error_data:
    final_results = pd.concat(all_error_data, ignore_index=True)
    print("\n=== Print11~20 error flow changereport ===")
    print(final_results)

if all_error_data:
    final_results = pd.concat(all_error_data, ignore_index=True)
    
    # reset index as timestamp (by error)
    error_indices = np.arange(len(final_results))
    
    # caculate sum correlation
    overall_pearson_corr, overall_pearson_p = stats.pearsonr(final_results['flow_change'], error_indices)
    overall_spearman_corr, overall_spearman_p = stats.spearmanr(final_results['flow_change'], error_indices)
    
    print("\n=== Print1~30 ===")
    print(final_results)
    
    # show sum correlation result
    print("\n=== Print1~30 ===")
    print(f"Print1~30 Pearson correlation: {overall_pearson_corr:.4f}, P-value: {overall_pearson_p:.4f}")
    print(f"Print1~30 Spearman correlation: {overall_spearman_corr:.4f}, P-value: {overall_spearman_p:.4f}")



1. Compare the distribution of filter flow changes during errors for each printer, including median, variability, and outliers.
2. Visually identify printers with abnormal flow change patterns or significantly different distributions for further analysis or maintenance consideration.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 8))
sns.boxplot(data=final_results, x='printer', y='flow_change', palette='viridis')

plt.title("Flow Change Distribution by Printer", fontsize=16)
plt.xlabel("Printer Number", fontsize=14)
plt.ylabel("Flow Change", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

1. Compare Pearson and Spearman correlation for each printer to evaluate how flow change relates to the error occurrence sequence.
2. Help identify machines with unusually high or low correlation, which may require further investigation or predictive maintenance actions.

In [ ]:
# Get every printer's correlation result（drop duplicates to avoid repeat error_df ）
correlation_summary = final_results[['printer', 'pearson_corr', 'spearman_corr']].drop_duplicates().sort_values('printer')

plt.figure(figsize=(16, 8))
bar_width = 0.35
x = range(len(correlation_summary))

plt.bar([i - bar_width/2 for i in x], correlation_summary['pearson_corr'], 
        width=bar_width, label='pearson_corr', color='steelblue')
plt.bar([i + bar_width/2 for i in x], correlation_summary['spearman_corr'], 
        width=bar_width, label='spearman_corr', color='seagreen')

plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title("Correlation by Printer", fontsize=16)
plt.xlabel("Printer Number", fontsize=14)
plt.ylabel("Correlation Coefficient", fontsize=14)
plt.xticks(x, correlation_summary['printer'])
plt.legend(title="correaltion type", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()